# SMILES-VAE — long training run on Colab (A100)

Trains the SMILES variational autoencoder on a GPU runtime using the best config from the 2026-07-25 tuning sweep (`scale_ll=0.1`, `latent=256`), on the Murcko-scaffold-balanced 50k subset of the 305k (no single scaffold/acyclic class dominates, so the VAE learns general chemistry instead of collapsing to a fatty-chain prior). On an A100 the full 50k / 30-epoch run finishes in well under an hour, versus ~7 h on the local CPU.

The notebook clones [`MauricioCafiero/SMILES_VAE`](https://github.com/MauricioCafiero/SMILES_VAE), installs deps, runs `code/run_train.py`, and shows the reconstruction / generation results.

**Run on a GPU runtime:** *Runtime → Change runtime type → GPU (A100).*

In [ ]:
!nvidia-smi

## 1. Clone the repo & install deps

In [ ]:
import os

BRANCH = "main"            # e.g. 'main' or a feature branch
REPO_DIR = "SMILES_VAE"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} https://github.com/MauricioCafiero/{REPO_DIR}.git

%cd {REPO_DIR}
!git checkout {BRANCH}
!git pull

# rdkit only — smiles_vae.py no longer imports transformers.
# TF, numpy, pandas, scikit-learn, matplotlib, Pillow are preinstalled on Colab.
!pip install -q rdkit

## 2. Configure & train

**Config cell** (next): defines the dataset + hyperparameters used by the training and `--load` cells. Run it before either. It does **not** train.

**Train cell** (after that): runs `code/run_train.py`. Skip it if you only want to re-run generation from saved weights — instead run the config cell, the inspect-results cell (to set `run_dir`), then the `--load` re-generation cell.

Defaults: scaffold-balanced 50k dataset, `scale_ll=0.1`, `latent=256`, 30 epochs. Set `NROWS>0` to subsample, `DATA` to point at another one-column SMILES CSV.

In [ ]:
# Config cell — defines the names the !python calls below interpolate
# ({DATA}, {NROWS}, {TEMPERATURE}, ...). Run this BEFORE training (next cell)
# AND before the --load re-generation cell (cell 11) so the names exist in the
# notebook namespace. This cell does NOT train.
DATA = "data/ZN305K_scaffold_balanced_50k.csv"   # one-column SMILES CSV
NROWS    = 0       # 0 = full dataset (50k). Set NROWS>0 to subsample for a quick test.
EPOCHS   = 30
GENERATE = 200
SCALE_LL = 0.1
LATENT   = 256
EMB      = 512
UNITS    = 256
LAYERS   = 1
# Autoregressive-decoder anti-collapse knobs (Bowman 2016):
WORD_DROPOUT_KEEP = 0.8   # 1.0 = off; lower to force latent z usage
ANNEAL_EPOCHS     = 10    # ramp scale_ll 0 -> target over this many epochs (0 = off)
# Decode temperature: 0.0 = greedy argmax (deterministic, mode-collapses); 0.7-1.0 = diverse sampling.
# For a post-training temp sweep, re-run only the --load re-gen cell with TEMPERATURE = 0.0 / 0.3 / 0.5.
TEMPERATURE = 0.7

# On GPU, cuDNN GRU + strict op-determinism can raise at fit time, so disable it.
%env SMILES_VAE_DISABLE_OP_DETERMINISM=1
!mkdir -p outputs

In [ ]:
# Train. SKIP this cell if you only want to re-run generation from saved
# weights (after a restart: run the config cell above + the inspect-results
# cell to set run_dir, then the --load re-generation cell).
!python code/run_train.py \
    --data {DATA} \
    --nrows {NROWS} --epochs {EPOCHS} --generate {GENERATE} \
    --scale_ll {SCALE_LL} --latent {LATENT} \
    --emb {EMB} --units {UNITS} --layers {LAYERS} \
    --word_dropout_keep {WORD_DROPOUT_KEEP} --anneal_epochs {ANNEAL_EPOCHS} \
    --temperature {TEMPERATURE} \
    2>&1 | tee outputs/long_run.log

## 3. Inspect the results

Prints the per-epoch history and the generated SMILES (standard `N(0,I)` vs empirical-latent sampling).

In [ ]:
import os, glob

runs = sorted(glob.glob('outputs/run_*'))
assert runs, 'no run dir found — did training finish?'
run_dir = runs[-1]
print('Latest run:', run_dir, '\n')

print('--- history.csv ---')
print(open(os.path.join(run_dir, 'history.csv')).read())

print('--- generated (empirical) ---')
print(open(os.path.join(run_dir, 'generated_smiles_empirical.txt')).read())

print('--- generated (standard N(0,I)) ---')
print(open(os.path.join(run_dir, 'generated_smiles_standard.txt')).read())

## 4. Reconstruction & generation grids

In [ ]:
from IPython.display import Image, display

for f in ['reconstruction_grid.png', 'generated_grid.png']:
    p = os.path.join(run_dir, f)
    if os.path.exists(p):
        print(p)
        display(Image(p))

## 5. Re-run generation without retraining

Uses `--load` to reuse the saved weights for a larger generation pass. No training happens — useful for sampling many more molecules once the model is trained.

In [ ]:
GEN_N = 500     # how many molecules to generate this pass

!python code/run_train.py --load {run_dir} \
    --data {DATA} \
    --nrows {NROWS} --generate {GEN_N} \
    --scale_ll {SCALE_LL} --latent {LATENT} \
    --emb {EMB} --units {UNITS} --layers {LAYERS} \
    --word_dropout_keep {WORD_DROPOUT_KEEP} \
    --temperature {TEMPERATURE}

## 6. Download the outputs

Zips all run artifacts (weights, config, history, generated SMILES, grids) and triggers a browser download.

In [ ]:
!zip -r -q smiles_vae_outputs.zip outputs/run_*
from google.colab import files
files.download('smiles_vae_outputs.zip')